# Phase 1

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import os
import shutil
import sys
import gymnasium as gym
import torch

torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from src.rl.env_adapter import MatchEnv
from src.rl.model import ActorCritic
from src.rl.pool import PoolOpponentController
from src.rl.ppo import train_mappo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
TEAM_SIZE = 3
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS_S3 = 3600   # 60s matches (360 decisions @ 6 Hz)
NUM_ENVS = 16

SAVE_DIR_S3 = "models/stage3/phase1"
POOL_DIR_S3 = os.path.join(SAVE_DIR_S3, "pool")
os.makedirs(POOL_DIR_S3, exist_ok=True)

STAGE2_BEST = "models/stage2/phase3/best_model.pt"
STAGE2_FINAL = "models/stage2/phase3/final_model.pt"

if os.path.exists(STAGE2_BEST):
    seed_weights = STAGE2_BEST
elif os.path.exists(STAGE2_FINAL):
    seed_weights = STAGE2_FINAL
else:
    raise FileNotFoundError("Missing Stage 2 Phase 3 model to warmstart Stage 3.")

shutil.copy(seed_weights, os.path.join(POOL_DIR_S3, "champion.pt"))
shutil.copy(seed_weights, os.path.join(POOL_DIR_S3, "history_0.pt"))
print(f"✅ Initialized Stage 3 pool with Stage 2 champion weights from: {seed_weights}")

✅ Initialized Stage 3 pool with Stage 2 champion weights from: models/stage2/phase3/best_model.pt


In [3]:
def make_s3_env(env_rank: int):
    def _thunk():
        # Curriculum: 5% Random, 25% Heuristic (4500 accel / 1500 kick), 70% Self-Play
        opp_ctrl = PoolOpponentController(
            pool_dir=POOL_DIR_S3,
            team="blue",
            device="cpu",
            p_random=0.15,
            p_heuristic=0.85,
        )
        env = MatchEnv(
            team_size=TEAM_SIZE,
            learner_team_size=TEAM_SIZE,
            opp_team_size=TEAM_SIZE,
            learner_team="red",
            max_round_steps=ROUND_STEPS_S3,
            action_repeat=10,         # 6 Hz control frequency
            heuristic_accel=3200.0,   
            heuristic_kick=1200.0,   
            goal_height=GOAL_H_REG,
            pitch_width=PITCH_W_REG,
            pitch_height=PITCH_H_REG,
            opponent_controller=opp_ctrl,
        )
        env.reset(seed=10000 + env_rank)
        return env
    return _thunk

envs_s3 = gym.vector.AsyncVectorEnv(
    [make_s3_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

In [4]:
model_s3 = ActorCritic().to(device)
ckpt = torch.load(seed_weights, map_location=device, weights_only=False)
state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt

model_s3.load_state_dict(state_dict, strict=True)
print(f"🔥 Successfully warmstarted 3v3 Model from: {seed_weights}")

train_mappo(
    envs=envs_s3,
    model=model_s3,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=50_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    update_epochs=3,
    minibatch_size=1536,                 # 256 * 16 * 3 = 12,288 step batch (8 minibatches)
    lr_init=5e-5,                        # Conservative fine-tuning LR
    lr_final=2e-6,
    ent_coef_init=0.008,
    ent_coef_final=0.0008,
    gamma=0.996,
    gae_lambda=0.97,
    active_tiers=["heuristic"],
    target_tier="heuristic",            
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    save_dir=SAVE_DIR_S3,
    pool_dir=POOL_DIR_S3,
    eval_episodes=50,                  
    eval_freq=250_000,
    max_steps=ROUND_STEPS_S3,
)

envs_s3.close()

🔥 Successfully warmstarted 3v3 Model from: models/stage2/phase3/best_model.pt
🚀 MAPPO Initialized | Format: 3v3 | Envs: 16 | Step Batch: 12288 | Device: cuda

📊 [EVALUATION @ Step 258,048 | SPS: 4883 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  58.0% | Reward: +1.312 | Goals: 47 Scored, 21 Conceded (+26 Net)
🏆 NEW CHAMPION REGISTERED @ step 258,048 -> models/stage3/phase1/pool/history_258048.pt
   ⭐⭐ PROMOTED! New Best Score (heuristic) -> [WR: 58.0%, Reward: +1.312, Net: +26]
      (Defeated previous record: [WR: None, Reward: None]) -> Saved: models/stage3/phase1/best_model.pt

📊 [EVALUATION @ Step 503,808 | SPS: 2946 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  56.0% | Reward: +1.880 | Goals: 60 Scored, 15 Conceded (+45 Net)
   ❌ Retaining current baseline. Did not pass criteria for heuristic: [WR: 58.0%, Reward: +1.312, Net: +26]

📊 [EVALUATION @ Step 761,856 | SPS: 2615 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  72.0% | Reward: +2.772

In [ ]:
from src.rl.visualization import evaluate_and_generate_html

best_s3_model = os.path.join(SAVE_DIR_S3, "best_model.pt")

evaluate_and_generate_html(
    red_agent=best_s3_model,
    blue_agent="heuristic",           # Red Champion vs Blue Champion
    red_team_size=3,
    blue_team_size=3,
    device=device,
    filename="stage3_selfplay_3v3.html",
    num_episodes=5,
    max_steps=3600,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    goal_height=GOAL_H_REG,
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/training_2/render/stage3_selfplay_3v3.html


'render/stage3_selfplay_3v3.html'

# Phase 2

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import shutil
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from src.rl.env_adapter import MatchEnv
from src.rl.model import ActorCritic
from src.rl.pool import PoolOpponentController
from src.rl.ppo import train_mappo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── 3v3 Match Parameters ──
TEAM_SIZE = 3
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS_P2 = 3600  # 60s matches 
NUM_ENVS = 16

SAVE_DIR_S3_P2 = "models/stage3/phase2"
POOL_DIR_S3_P2 = os.path.join(SAVE_DIR_S3_P2, "pool")
os.makedirs(POOL_DIR_S3_P2, exist_ok=True)

# Warmstart from Stage 3 Phase 1 Best/Final Model
phase1_best = "models/stage3/phase1/best_model.pt"
phase1_final = "models/stage3/phase1/final_model.pt"

if os.path.exists(phase1_best):
    seed_weights = phase1_best
elif os.path.exists(phase1_final):
    seed_weights = phase1_final
else:
    raise FileNotFoundError("Missing Stage 3 Phase 1 model to warmstart Phase 2.")

shutil.copy(seed_weights, os.path.join(POOL_DIR_S3_P2, "champion.pt"))
shutil.copy(seed_weights, os.path.join(POOL_DIR_S3_P2, "history_0.pt"))
print(f"🔥 Seeded Phase 2 pool with Phase 1 weights from: {seed_weights}")


def make_s3_p2_env(env_rank: int):
    def _thunk():
        opp_ctrl = PoolOpponentController(
            pool_dir=POOL_DIR_S3_P2,
            team="blue",
            device="cpu",
            p_random=0.05,       
            p_heuristic=0.15,   
        )
        env = MatchEnv(
            team_size=TEAM_SIZE,
            learner_team_size=TEAM_SIZE,
            opp_team_size=TEAM_SIZE,
            learner_team="red",
            max_round_steps=ROUND_STEPS_P2,
            action_repeat=10,
            heuristic_tiers=[
                (3200.0, 1200.0),  
                # (3400.0, 1500.0), 
                # (4000.0, 1600.0),  
                # (4200.0, 1700.0),  
            ],
            goal_height=GOAL_H_REG,
            pitch_width=PITCH_W_REG,
            pitch_height=PITCH_H_REG,
            opponent_controller=opp_ctrl,
        )
        env.reset(seed=12000 + env_rank)
        return env
    return _thunk

envs_s3_p2 = gym.vector.AsyncVectorEnv(
    [make_s3_p2_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

model_s3_p2 = ActorCritic().to(device)
ckpt = torch.load(seed_weights, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s3_p2.load_state_dict(state_dict, strict=True)
print("✅ 3v3 Model weights loaded successfully.")

Using device: cuda
🔥 Seeded Phase 2 pool with Phase 1 weights from: models/stage3/phase1/best_model.pt
✅ 3v3 Model weights loaded successfully.


In [ ]:
train_mappo(
    envs=envs_s3_p2,
    model=model_s3_p2,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=55_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    update_epochs=3,
    minibatch_size=1024,
    lr_init=3e-5,
    lr_final=2e-6,
    ent_coef_init=0.008,
    ent_coef_final=0.0008,
    gamma=0.994,
    gae_lambda=0.95,
    active_tiers=["heuristic", "champion"],
    target_tier="champion",
    filter_thresholds={"heuristic": 0.80},
    eval_episodes=70,                                      # Total 70 episodes
    tier_ratios={"heuristic": 0.50, "champion": 0.50},     # 35 vs Heuristic, 35 vs Champion
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    save_dir=SAVE_DIR_S3_P2,
    pool_dir=POOL_DIR_S3_P2,
    eval_freq=250_000,
    max_steps=ROUND_STEPS_P2,
    action_repeat=10,
)

envs_s3_p2.close()

🚀 Accelerated MAPPO Initialized | Format: 3v3 | Envs: 16 | Batch: 12288 | Device: cuda

📊 [EVALUATION @ Step 258,048 | Rollout SPS: 3178 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  91.4% | Reward: +3.246 | Goals: 82 Scored, 6 Conceded (+76 Net)
   ⚔️  vs Champion  [TARGET] | WR:  40.0% | Reward: +0.311 | Goals: 19 Scored, 10 Conceded (+9 Net)
🏆 NEW CHAMPION REGISTERED @ step 258,048 -> models/stage3/phase2/pool/history_258048.pt
   ⭐⭐ PROMOTED! New Best Score (champion) -> [WR: 40.0%, Reward: +0.311, Net: +9]
      (Defeated previous record: [WR: 40.0%, Reward: +0.311]) -> Saved: models/stage3/phase2/best_model.pt (Eval took 40.0s)

📊 [EVALUATION @ Step 503,808 | Rollout SPS: 3328 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  94.3% | Reward: +3.146 | Goals: 74 Scored, 3 Conceded (+71 Net)
   ⚔️  vs Champion  [TARGET] | WR:  28.6% | Reward: -0.260 | Goals: 15 Scored, 16 Conceded (-1 Net)
   ❌ Retaining current baseline. Did not pass 

In [ ]:
from src.rl.visualization import evaluate_and_generate_html

best_s3_p2_model = os.path.join(SAVE_DIR_S3_P2, "best_model.pt")

evaluate_and_generate_html(
    red_agent=best_s3_p2_model,
    blue_agent="models/stage3/phase1/best_model.pt",           # Red Champion vs Blue Champion
    red_team_size=3,
    blue_team_size=3,
    device=device,
    filename="stage3_phase2_selfplay_3v3.html",
    num_episodes=10,
    max_steps=3600,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    goal_height=GOAL_H_REG,
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/training_2/render/stage3_phase2_selfplay_3v3.html


'render/stage3_phase2_selfplay_3v3.html'